### Ingestión del archivo "genre.csv"
#### Este notebook es un clon del 02-Ingestion File genre de la carpeta ingestion con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
2. Se modifica la creación del genre_df, usando la variable del contenedor de bronze incremental y apuntando al widget v_file_date
3. Se agrega la columna file_date
4. Se elimina el paso "Escribir datos en el datalake en formato Parquet"
5. Se cambia el esquema movie_silver por movie_silver_inc a la hora de crear la managed table

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark
### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
genre_schema = "genreId INT, genreName STRING"






In [0]:
genre_df = spark.read.option("header",True).schema(genre_schema).csv(f"{bronze_inc_folder_path}/{v_file_date}/genre.csv")



In [0]:
genre_df.printSchema()

In [0]:
display(genre_df)

### Paso 2 - Cambiar el nombre de las columnas según lo requerido

In [0]:
genre_renamed_of = genre_df \
                        .withColumnRenamed("genreId", "genre_id") \
                        .withColumnRenamed("genreName", "genre_name")



                        



In [0]:
display(genre_renamed_of)

### Paso 3 - Agregar la columnas "ingestion_date", "environment" y "file_date" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit











In [0]:
#genre_final_df = genre_renamed_of.withColumn("ingestion_date", current_timestamp()) \
#                                    .withColumn("env", lit("production"))

genre_final_df = add_ingestion_date(genre_renamed_of) \
                    .withColumn("environment", lit(v_environment)) \
                    .withColumn("file_date", lit(v_file_date))                                   
                                  











                                    

In [0]:
display(genre_final_df)

### Paso 4 - Escribir datos en una managed table en el contenedor silver

In [0]:
genre_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver_inc.genres")

In [0]:
%sql
SELECT * FROM movie_silver_inc.genres

In [0]:
dbutils.notebook.exit("Success")